## tl;dr

**Observed clock support, not profitability.**

- case: 38/63 eligible; 10 pending and 0 unknown.
- control: 65/108 eligible; 19 pending and 0 unknown.

All 171 original requests remain in the denominators. No economic outcomes were calculated.


## Context & Methods

This is a saved-output clock audit, not a backtest or a fill simulation.

### Key Assumptions

Original case/control requests and fixed reference stops are unchanged. The one-hour observation limit is administrative, not an entry expiry.

Prefix comparisons normalize opaque segment labels; both paths share the SMA implementation, so this is not independent formula verification.

Three code cells run with stdlib sequential exec and captured stdout, not a Jupyter kernel or nbformat certification.


In [1]:
# tl;dr / Context & Methods
# Saved V38 audit only. Eligibility is not a fill or profit.
# Key assumptions: original 63/108 requests, fixed stops, 60-minute administrative observation.
import json
from yoyo.evaluation import owner_k1k2_pending_entry_report as report
commit, summary, frames, hashes = report.guard()
print(json.dumps({'scope': 'V38 saved outputs only; no market/outcome reads', 'source_hashes': hashes}, indent=2))


{
  "scope": "V38 saved outputs only; no market/outcome reads",
  "source_hashes": {
    "experiments/active/exp-btcusdtp-owner-k1k2-pending-entry-20260907-v38/config.json": "dd39bcd558c2396a7ee72259743bc2c1f30289dc8759e5db45647d93c8cb5e41",
    "experiments/active/exp-btcusdtp-owner-k1k2-pending-entry-20260907-v38/PROJECT_PLAN.md": "eef36617af1d8e6036e24be91ea2383efdc801e4ee00955ee8eee1b931cbb3f2",
    "tests/test_k1k2_pending_entry.py": "0e124f4a38f1bc5ba3038826f2ed2992ebb4ad173ead5f134dcbb2709fd2a56a",
    "tests/test_owner_k1k2_pending_entry_audit.py": "d607c2d43ddf3b4e8e408cfd5f0dd41e1127b4f5ac5404754408e92eef4456f1",
    "yoyo/data/hourly_impulse.py": "f1128f514456f0e1a7bf89b719b2333ebe3b12f66402d58c82764d355be97a3b",
    "yoyo/data/k1k2_genuine_flow_alignment.py": "7ffa25c6fadcc9479d87280e7085c3a370bb201fecd13420728070a91b5cec83",
    "yoyo/data/k1k2_pending_entry.py": "0cfc3c27c83351e7109fe360e841718845f37b173f98dc723abb1ace9bf20fb9",
    "yoyo/evaluation/owner_k1k2_genuine_flo

## Data

Only the eight hash-verified V38 saved CSVs are read. The next cell executes STATUS_SQL on every original event, without dropping unknown or pending observations.


In [2]:
# Data / Results: execute the exact STATUS_SQL on all saved observations.
counts = report.execute_counts(frames, summary)
print(report.STATUS_SQL)
print(counts.to_json(orient='records', indent=2))


WITH totals AS (SELECT cohort,COUNT(*) AS denominator FROM main.audit_events GROUP BY cohort)
SELECT a.cohort,a.status,COUNT(*) AS events,t.denominator,1.0*COUNT(*)/t.denominator AS share,
SUM(a.status='eligible' AND a.waiting_bars=0) AS immediately_eligible,
SUM(a.status='eligible' AND a.waiting_bars>0) AS delayed_eligible,
AVG(a.waiting_bars*5.0) AS observed_wait_minutes
FROM main.audit_events a JOIN totals t ON a.cohort=t.cohort
GROUP BY a.cohort,a.status,t.denominator ORDER BY a.status,a.cohort
[
  {
    "cohort":"case",
    "status":"eligible",
    "events":38,
    "denominator":63,
    "share":0.6031746032,
    "immediately_eligible":16,
    "delayed_eligible":22,
    "observed_wait_minutes":9.4736842105
  },
  {
    "cohort":"control",
    "status":"eligible",
    "events":65,
    "denominator":108,
    "share":0.6018518519,
    "immediately_eligible":27,
    "delayed_eligible":38,
    "observed_wait_minutes":14.4615384615
  },
  {
    "cohort":"case",
    "status":"invalidated_

## Results

The grouped rows above retain event count, original cohort denominator, share, immediate/delayed eligibility, and observed waiting minutes. Shares are fractions from 0 to 1, not returns.


In [3]:
# Takeaways: preserve unknown/pending, no cash-zero or return inference.
assert counts.events.sum() == 171
assert summary['prefix_checks'] == {'events': 171, 'all_passed': True}
print(json.dumps({'requests': 171, 'case': 63, 'control': 108, 'prefix_passed': 171, 'status_counts': counts.to_dict('records'), 'economic_outcomes': False, 'interpretation': 'Eligibility and waiting clocks only; pending is right-censored, unknown remains unknown.'}))


{"requests": 171, "case": 63, "control": 108, "prefix_passed": 171, "status_counts": [{"cohort": "case", "status": "eligible", "events": 38, "denominator": 63, "share": 0.6031746031746031, "immediately_eligible": 16, "delayed_eligible": 22, "observed_wait_minutes": 9.473684210526315}, {"cohort": "control", "status": "eligible", "events": 65, "denominator": 108, "share": 0.6018518518518519, "immediately_eligible": 27, "delayed_eligible": 38, "observed_wait_minutes": 14.461538461538462}, {"cohort": "case", "status": "invalidated_wait_bar", "events": 15, "denominator": 63, "share": 0.23809523809523808, "immediately_eligible": 0, "delayed_eligible": 0, "observed_wait_minutes": 20.666666666666668}, {"cohort": "control", "status": "invalidated_wait_bar", "events": 24, "denominator": 108, "share": 0.2222222222222222, "immediately_eligible": 0, "delayed_eligible": 0, "observed_wait_minutes": 21.875}, {"cohort": "case", "status": "pending_at_cutoff", "events": 10, "denominator": 63, "share": 0.

## Takeaways

Eligibility is not an executed trade or a profitable trade. Unknown remains unknown; pending is administrative right-censoring and cannot be assigned cash-zero return. A separately frozen economic policy is still required.
